# E791 $D^+\to\pi^-\pi^+\pi^+$ — coefficient-fit closure test

This notebook performs the first clean closure test of the fitter.

The procedure is deliberately simple:

1. define the E791 Fit-2 amplitude model with known coefficients;
2. keep all resonance masses, widths, spins and meson radii fixed;
3. generate one pseudo-data sample from the known model;
4. randomize the initial values of the free complex coefficients;
5. perform **one** unbinned maximum-likelihood fit;
6. compare the fitted coefficients with the injected values.

The $\rho(770)\pi^+$ coefficient is fixed to $1+0i$ and defines the global magnitude/phase convention. All other coefficients are fitted in Cartesian form $(x,y)$.

No multistart procedure is used here. That will be tested separately later.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzGrid,
    DecayChannel,
    DecayModel,
    Minimizer,
    NonResonant,
    Parameter,
    RealImag,
    Resonance,
    enable_x64,
    weighted_resample,
)

enable_x64()


## 1. E791 Fit-2 truth model

We use the same numerical values as the E791 generation example. The internal convention includes the previously discussed $180^\circ$ shift of the non-resonant coefficient.

All lineshape parameters are ordinary fixed numbers in this notebook — **not** `Parameter` objects.


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma":   (1.17, 205.7),
    "rho770":  (1.00,   0.0),
    "NR":      (0.48,  57.3),
    "f0_980":  (0.43, 165.0),
    "f2_1270": (0.76,  57.3),
    "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def polar_to_xy(r, phase_deg):
    phase = np.deg2rad(phase_deg)
    return r * np.cos(phase), r * np.sin(phase)

def internal_xy(name):
    r, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    return polar_to_xy(r, phase)

truth_xy = {name: internal_xy(name) for name in fit2_polar}

print(f"{'component':10s} {'x truth':>12s} {'y truth':>12s}")
for name, (x, y) in truth_xy.items():
    print(f"{name:10s} {x:12.6f} {y:12.6f}")


## 2. Build a model with only the coefficients free

The truth values are stored separately from the initial values of the fit parameters. This prevents the fitter from obtaining any truth information from the `Parameter` defaults.

The $\rho(770)$ coefficient remains fixed at $1+0i$. The floating Cartesian coefficients have no Minuit bounds.


In [ ]:
truth = {}

def free_coefficient(name):
    x_truth, y_truth = truth_xy[name]
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)

    return RealImag(
        Parameter.coefficient(
            f"{name}.x",
            0.0,
            owner=name,
            step=0.01,
        ),
        Parameter.coefficient(
            f"{name}.y",
            0.0,
            owner=name,
            step=0.01,
        ),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}

components = [
    Resonance("sigma",   (0, 1), coefficients["sigma"],   mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770",  (0, 1), coefficients["rho770"],  mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980",  (0, 1), coefficients["f0_980"],  mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0, 1), coefficients["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0, 1), coefficients["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0, 1), coefficients["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]

model = DecayModel(channel, components)

print("Free fit parameters:")
for p in model.parameters:
    if not p.fixed:
        print(f"  {p.name:16s} bounds={p.bounds}")

print("\nNumber of free parameters:", sum(not p.fixed for p in model.parameters))


## 3. Deterministic normalization grid

For this first closure test we keep the normalization setup simple and deterministic. The same fixed `DalitzGrid` is used to normalize the truth model during generation and the fitted model in the likelihood.

The adaptive-grid study is kept separate so that this test isolates the coefficient fitting machinery.


In [ ]:
GRID_N = 1000

norm = DalitzGrid(
    channel.parent_mass,
    channel.daughter_masses,
    resolution=GRID_N,
).sample()

print(f"normalization grid: {GRID_N} x {GRID_N}")
print(f"grid points       : {norm.size:,}")
print(f"weight spread     : {float(jnp.ptp(norm.weights)):.3e}")


## 4. Generate pseudo-data from the known coefficients

An independent phase-space pool is generated and reweighted with the truth intensity. The pool is used only to draw the unweighted toy sample.


In [ ]:
N_POOL = 1_000_000
N_DATA = 100_000

pool = model.generate_phase_space(N_POOL, seed=2000)

truth_cache_pool = model.prepare_cache(pool, norm)
truth_intensity, truth_normalization = truth_cache_pool.evaluate(truth)
target_weights = pool.weights * truth_intensity

data = weighted_resample(
    jax.random.key(791),
    pool,
    target_weights,
    N_DATA,
    replace=True,
)

print(f"candidate pool       : {pool.size:,}")
print(f"pseudo-data events   : {data.size:,}")
print(f"truth normalization  : {float(truth_normalization):.12g}")
print("finite intensity     :", bool(jnp.all(jnp.isfinite(truth_intensity))))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
h = ax.hist2d(
    np.asarray(data.s12),
    np.asarray(data.s13),
    bins=110,
)
fig.colorbar(h[3], ax=ax, label="events")
ax.set_xlabel(r"$s_{12}$ [GeV$^2$]")
ax.set_ylabel(r"$s_{13}$ [GeV$^2$]")
ax.set_title(r"E791 Fit-2 pseudo-data: $D^+\to\pi^-\pi^+\pi^+$")
plt.show()


## 5. Build the likelihood and randomize one starting point

The coefficient fit itself is **unbounded**. The interval below is used only to draw the initial point; it is not passed to Minuit as a parameter limit.

This deliberately tests whether one minimization can recover the physical solution from a broad starting point.


In [ ]:
cache = model.prepare_cache(data, norm)

def nll(values):
    intensity, normalization = cache.evaluate(values)
    return (
        -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300)))
        + data.size * jnp.log(normalization)
    )

minimizer = Minimizer(
    nll,
    model.parameters,
    verbose=1,
)

START_SEED = 314159
START_RANGE = (-2.5, 2.5)
rng = np.random.default_rng(START_SEED)
start_values = {
    p.name: float(rng.uniform(*START_RANGE))
    for p in model.parameters
    if not p.fixed
}

print(f"Random start seed  = {START_SEED}")
print(f"Start draw interval = {START_RANGE} (NOT a fit limit)")
print(f"{'parameter':16s} {'truth':>11s} {'start':>11s} {'delta':>11s}")
for p in model.parameters:
    if p.fixed:
        continue
    t = truth[p.name]
    s = start_values[p.name]
    print(f"{p.name:16s} {t:11.6f} {s:11.6f} {s-t:+11.6f}")

print(f"\nNLL(truth) = {float(nll(truth)):.6f}")
print(f"NLL(start) = {float(nll(start_values)):.6f}")


## 6. Perform exactly one fit

This is intentionally **not** a multistart fit. The fitter runs one MIGRAD refinement followed by HESSE.


In [ ]:
result = minimizer.fit(
    start_values=start_values,
    simplex=False,
)

fit_values = {
    p.name: float(result.values[p.name])
    for p in model.parameters
    if not p.fixed
}

print("\nFit summary")
print("-----------")
print("valid            :", bool(result.valid))
print("NLL(start)       :", float(nll(start_values)))
print("NLL(truth)       :", float(nll(truth)))
print("NLL(fit)         :", float(result.fval))
print("NLL(fit)-truth   :", float(result.fval - nll(truth)))
print("EDM              :", float(result.fmin.edm))
print("function calls   :", int(result.nfcn))


## 7. Closure table

For a finite toy sample, the fitted parameters are not expected to land exactly at the injected values. The useful diagnostic is whether the deviations are statistically compatible with the fitted uncertainties.


In [ ]:
rows = []

print(
    f"{'parameter':16s} {'truth':>10s} {'start':>10s} "
    f"{'fit':>10s} {'error':>10s} {'pull':>9s}"
)

for p in model.parameters:
    if p.fixed:
        continue

    t = float(truth[p.name])
    s = float(start_values[p.name])
    f = float(result.values[p.name])
    e = float(result.errors[p.name])
    pull = (f - t) / e

    rows.append((p.name, t, s, f, e, pull))
    print(f"{p.name:16s} {t:10.5f} {s:10.5f} {f:10.5f} {e:10.5f} {pull:9.3f}")


## 8. Cartesian coefficients: truth vs start vs fit


In [ ]:
names = [p.name for p in model.parameters if not p.fixed]
xpos = np.arange(len(names))
truth_array = np.array([truth[name] for name in names])
start_array = np.array([start_values[name] for name in names])
fit_array = np.array([result.values[name] for name in names], dtype=float)
fit_error = np.array([result.errors[name] for name in names], dtype=float)

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.scatter(xpos, truth_array, marker="x", s=70, label="truth")
ax.scatter(xpos, start_array, marker="o", s=35, label="random start")
ax.errorbar(
    xpos,
    fit_array,
    yerr=fit_error,
    fmt=".",
    capsize=3,
    label="fit",
)
ax.axhline(0.0, linewidth=0.8)
ax.set_xticks(xpos)
ax.set_xticklabels(names, rotation=60, ha="right")
ax.set_ylabel("Cartesian coefficient")
ax.set_title("E791 coefficient closure: truth, random start and fitted value")
ax.legend()
fig.tight_layout()
plt.show()


## 9. Convert the coefficients back to magnitude and phase

This representation is easier to compare with the conventional amplitude-analysis tables. The internal NR phase is shifted back by $180^\circ$ for display so that the numbers can be compared directly with the published E791 convention.


In [ ]:
component_order = [
    "sigma",
    "NR",
    "f0_980",
    "f2_1270",
    "f0_1370",
    "rho1450",
]

def xy_to_polar(x, y, name):
    magnitude = np.hypot(x, y)
    phase = np.rad2deg(np.arctan2(y, x)) % 360.0
    if name == "NR":
        phase = (phase - 180.0) % 360.0
    return magnitude, phase

print(f"{'component':10s} {'mag truth':>11s} {'mag start':>11s} {'mag fit':>11s} {'phase truth':>12s} {'phase start':>12s} {'phase fit':>12s}")

for name in component_order:
    tx, ty = truth[f"{name}.x"], truth[f"{name}.y"]
    sx, sy = start_values[f"{name}.x"], start_values[f"{name}.y"]
    fx, fy = result.values[f"{name}.x"], result.values[f"{name}.y"]

    mt, pt = xy_to_polar(float(tx), float(ty), name)
    ms, ps = xy_to_polar(float(sx), float(sy), name)
    mf, pf = xy_to_polar(float(fx), float(fy), name)

    print(
        f"{name:10s} {mt:11.4f} {ms:11.4f} {mf:11.4f} "
        f"{pt:12.2f} {ps:12.2f} {pf:12.2f}"
    )


## 10. Projection before and after the fit

The independent phase-space pool is reused only as a high-statistics plotting sample. It is evaluated at the randomized starting coefficients, at the fitted coefficients and at truth.

The likelihood itself still uses the deterministic normalization grid.


In [ ]:
projection_cache = model.prepare_cache(pool, norm)

def projection(values, bins):
    intensity, _ = projection_cache.evaluate(values)
    weights = np.asarray(pool.weights * intensity)

    h12, _ = np.histogram(np.asarray(pool.s12), bins=bins, weights=weights)
    h13, _ = np.histogram(np.asarray(pool.s13), bins=bins, weights=weights)
    return h12 + h13

data_projection = np.concatenate([np.asarray(data.s12), np.asarray(data.s13)])
bins = np.linspace(data_projection.min(), data_projection.max(), 110)
centers = 0.5 * (bins[:-1] + bins[1:])

h_data, _ = np.histogram(data_projection, bins=bins)
h_start = projection(start_values, bins)
h_fit = projection(fit_values, bins)
h_truth = projection(truth, bins)

for h in (h_start, h_fit, h_truth):
    h *= h_data.sum() / h.sum()

fig, ax = plt.subplots(figsize=(10.5, 6))
ax.errorbar(
    centers,
    h_data,
    yerr=np.sqrt(np.maximum(h_data, 1)),
    fmt=".",
    label="pseudo-data",
)
ax.step(centers, h_start, where="mid", linewidth=1.3, label="random start")
ax.step(centers, h_fit, where="mid", linewidth=1.8, label="single fit")
ax.step(centers, h_truth, where="mid", linestyle="--", linewidth=1.4, label="truth")
ax.set_xlabel(r"$m^2(\pi^-\pi^+)$ [GeV$^2$]")
ax.set_ylabel("entries / bin")
ax.set_title("Projection before and after the coefficient fit")
ax.legend()
plt.show()


## 11. Pull summary

This is a single pseudo-experiment, so the pull distribution itself is not expected to be Gaussian. The purpose here is only to identify obvious closure failures. A statistically meaningful pull study requires many toys and will be implemented later.


In [ ]:
pulls = np.array([row[-1] for row in rows])

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.axhline(0.0, linewidth=1.0)
ax.axhline(+1.0, linestyle="--", linewidth=0.9)
ax.axhline(-1.0, linestyle="--", linewidth=0.9)
ax.scatter(np.arange(len(names)), pulls, s=45)
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=60, ha="right")
ax.set_ylabel(r"$(\hat\theta-\theta_{true})/\sigma$")
ax.set_title("Single-toy coefficient pulls")
fig.tight_layout()
plt.show()

print(f"max |pull| = {np.max(np.abs(pulls)):.3f}")
print(f"RMS pull   = {np.sqrt(np.mean(pulls**2)):.3f}")


## What this test establishes

If this notebook closes successfully, it tests the complete coefficient-only chain:

```text
known amplitude model
      -> toy generation
      -> randomized coefficient start
      -> cached component amplitudes
      -> coherent normalization
      -> unbinned NLL
      -> JAX gradients
      -> one Minuit fit
      -> recovered coefficients
```

Masses and widths remain fixed throughout. Only after this basic closure test is satisfactory should we reintroduce dynamic parameters or multistart minimization.
